# Mandatory assignment 3

**GitHub repository:** *Insert the URL to your repository here.* https://github.com/IbenSjoblom-prog/TECH2-H26-assignment3 

## About the assignment

- Complete the assignment individually.
- You may use online resources, including generative AI. Include a statement
  explaining how you used AI, or state that you did not use it.

## Submission

- Submit your completed `assignment3.ipynb` on Canvas by
  **October 23, 2026, at 23:59**.

## Submission requirements

- Create a new repository from the template linked on Canvas (click **Use this
  template** > **Create a new repository**) and implement your solution in your
  GitHub repository.
- Use the TECH2 environment from the course.
- Implement your solution directly in this notebook by inserting code and text
  cells. Do not create additional notebooks or files.
- Include all requested numerical results, figures, and short written answers.
- Give each figure a title, label its axes, and include a legend where needed.
- Before you submit, make sure your GitHub repository is **publicly accessible** and include its URL at the
  top of this notebook.
- Before submission, restart the kernel and run all cells in order. Your notebook
  must run without errors and find the CSV without an absolute file path.

## Hints

- Use the material from lectures and workshops 1–3: functions, loops, NumPy,
  pandas, and Matplotlib.
- Compute group statistics using loops; there is no need to use `groupby()`
  (which we cover in lecture 4), but you can do so if you prefer.

***
## Stock market participation in US households

Who participates in the stock market, and how has participation changed over time?
To answer this question, you will analyze data from the
[Survey of Consumer Finances](https://www.federalreserve.gov/econres/scfindex.htm)
(SCF), a representative survey of US households conducted every three years
from 1989 to 2022.

The file `SCF_10pct.csv` contains a 10% sample of the SCF data. The central variables of
interest for this assignment are:

1. `stocks`, which reports the value of directly held stocks (in US dollars); and
2. `equity`, which reports the value of all equity holdings (in US dollars),
   including direct stock ownership and indirect exposure through mutual funds,
   retirement accounts, and other investment vehicles.

We measure stock market participation in two ways:

- **Direct participation:** a household owns individual stocks (`stocks > 0`).
- **Overall participation:** a household holds equity directly or indirectly
  (`equity > 0`).

For each measure, the *participation rate* is the percentage of households
satisfying that condition. We therefore calculate two rates: the **direct
participation rate** and the **overall participation rate**. Direct participants
are included among overall participants, so the two rates should not be added.

## Data description

| Variable | Description |
|----------|-------------|
| `id` | Unique observation identifier |
| `year` | Survey year |
| `age` | Age of the household reference person |
| `educ` | Education level: `1`: No high school; `2`: High school; `3`: Some college; `4`: Bachelor's or higher |
| `stocks` | Value of directly held individual stocks (in US dollars) |
| `equity` | Total directly and indirectly owned stock equity (in US dollars) |

***
## Part 1 — Data preparation

1. Load `SCF_10pct.csv` into a pandas DataFrame. Print the columns contained in
   the DataFrame and their number of non-missing values.
2. Drop all observations that have missing values for any of the variables listed
   in the table above. For each variable, report how many observations you dropped
   and how many remain.
3. Keep observations with `25 <= age <= 89`, including both endpoints.
   Report how many observations you dropped and how many remain.
4. Create two 0/1 indicator columns:
   - `direct`: 1 if the household participates directly (`stocks > 0`),
     0 otherwise.
   - `overall`: 1 if the household participates directly or indirectly
     (`equity > 0`), 0 otherwise.

### Solutions Part 1 

### Task 1. 

In [25]:
import pandas as pd

# loading survey data
scf = pd.read_csv("SCF_10pct.csv")

# Display each column and its number of non-missing values
print(scf.count())

id        5808
year      5808
age       5808
educ      5792
stocks    5790
equity    5792
dtype: int64


### Task 2.

In [26]:
columns_needed = ["id", "year", "age", "educ", "stocks", "equity"]
starting_rows = len(scf)

for column in columns_needed:
    rows_before = len(scf)
    scf = scf.dropna(subset=[column])
    print(f"{column}: {rows_before - len(scf)} dropped; {len(scf)} remain")

print(f"Total dropped: {starting_rows - len(scf)}")

id: 0 dropped; 5808 remain
year: 0 dropped; 5808 remain
age: 0 dropped; 5808 remain
educ: 16 dropped; 5792 remain
stocks: 18 dropped; 5774 remain
equity: 16 dropped; 5758 remain
Total dropped: 50


### Task 3. 

In [27]:
rows_before = len(scf)
scf = scf[scf["age"].between(25, 89)]
rows_after = len(scf)

print(f"Removed {rows_before - rows_after} observations; {rows_after} remain.")

Removed 326 observations; 5432 remain.


### Task 4

In [28]:
scf["direct"] = scf["stocks"].gt(0).astype(int)
scf["overall"] = scf["equity"].gt(0).astype(int)

scf.head()

,id,year,age,educ,stocks,equity,direct,overall
0,0,1989,35,2.0,0.0,0.0,0,0
1,1,1989,33,2.0,0.0,0.0,0,0
2,2,1989,62,1.0,0.0,0.0,0,0
4,4,1989,70,1.0,0.0,0.0,0,0
5,5,1989,50,1.0,0.0,0.0,0,0


***
## Part 2 — Participation rates

You are now ready to compute the stock market participation rates for the
final sample. You will compute these in two ways: first, by writing a function
that uses a loop to compute the participation rate, and second, by using pandas's
built-in `mean()` method.

1. Write a function `participation_rate(indicator)` that takes a nonempty
   one-dimensional sequence of 0/1 values and returns the fraction of ones
   (in percent).
2. Apply your function to the final sample's `direct` and `overall`
   columns to compute the direct and overall participation rates. Report the
   results.
3. Use pandas's built-in `mean()` method to compute the direct and overall
   participation rates. Report the results.

### Part 2 

### Task 1 

In [29]:
def participation_rate(indicator):
    total = 0

    for value in indicator:
        total += value

    return round(100 * total / len(indicator), 1)

### Task 2 

In [30]:
print(f"Direct participatio rate: {participation_rate(scf['direct'])}%")
print(f"Overall participation rate: {participation_rate(scf['overall'])}%")

Direct participatio rate: 17.0%
Overall participation rate: 49.4%


### Task 3

In [31]:
direct_rate = 100 * scf["direct"].mean()
overall_rate = 100 * scf["overall"].mean()

print(f"Direct participation rat: {direct_rate:.1f}%")
print(f"Overall participation rate: {overall_rate:.1f}%")

Direct participation rat: 17.0%
Overall participation rate: 49.4%


***
## Part 3 — Participation by education

1. Loop over education categories 1–4. For each category, calculate the direct
   participation rate and the overall participation rate (in percent).
2. Display the results, rounding rates to one decimal place.
3. Create a figure with two subplots, one for each rate, containing bar charts
   of participation rates across education categories.
4. In two or three sentences, describe the education pattern.

### Part 3 

### 1 

In [32]:
direct_rates = []
overall_rates = []

for level in range(1, 5):
    households = scf[scf["educ"] == level]
    direct_rates.append(participation_rate(households["direct"]))
    overall_rates.append(participation_rate(households["overall"]))

### 2 

In [33]:
for level, direct, overall in zip(range(1, 5), direct_rates, overall_rates):
    print(f"Education level {level}: direct {direct:.1f}%, overall {overall:.1f}%")




Education level 1: direct 4.4%, overall 17.3%
Education level 2: direct 11.4%, overall 39.9%
Education level 3: direct 13.5%, overall 48.3%
Education level 4: direct 30.5%, overall 73.5%


### 3 

In [34]:
years = sorted(scf["year"].unique())
direct_by_year = []
overall_by_year = []

for year in years:
    households = scf[scf["year"] == year]
    direct_by_year.append(participation_rate(households["direct"]))
    overall_by_year.append(participation_rate(households["overall"]))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(years, direct_by_year, marker="o", label="Direct")
ax.plot(years, overall_by_year, marker="o", label="Overall")
ax.set_title("Stock-market participation over time")
ax.set_xlabel("Year")
ax.set_ylabel("Participation rate (%)")
ax.legend()
fig.tight_layout()
plt.show()
# I could not verify this plot because the Jupyter kernel crashed when I ran the cell.

NameError: name 'plt' is not defined

***
## Part 4 — Participation over time

1. Loop over years. For each year, compute the direct participation rate and the
   overall participation rate.
2. Create a line plot showing both participation rates over time in a single figure.
3. Report the changes from 1989 to 2022 in the direct and overall participation
   rates, both in percentage points.
4. Describe the main time patterns in two or three sentences.

### Task 1 

In [ ]:
years = sorted(scf["year"].unique())
direct_rates_y = []
overall_rates_y = []
for year in years:
    year_data = scf[scf["year"] == year]
    direct_rates_y.append(participation_rate(year_data["direct"]))
    overall_rates_y.append(participation_rate(year_data["overall"]))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(years, direct_rates_y, marker="o", label="Direct")
ax.plot(years, overall_rates_y, marker="o", label="Overall")

ax.set_title("Yearly stock-market participation")
ax.set_xlabel("Survey year")
ax.set_ylabel("Participation rate (%)")
ax.legend()

fig.tight_layout()
#Difficult to do these tasks when the kernel crashed,

NameError: name 'plt' is not defined

In [ ]:
for i in range(len(years)):
    print(years[i], "Direct:", direct_rates_y[i], "%",
          "Overall:", overall_rates_y[i], "%")


1989 Direct: 15.6 % Overall: 31.6 %
1992 Direct: 18.3 % Overall: 40.1 %
1995 Direct: 15.7 % Overall: 45.9 %
1998 Direct: 19.9 % Overall: 51.2 %
2001 Direct: 21.8 % Overall: 57.0 %
2004 Direct: 18.9 % Overall: 54.7 %
2007 Direct: 17.3 % Overall: 56.5 %
2010 Direct: 17.4 % Overall: 50.3 %
2013 Direct: 14.8 % Overall: 49.9 %
2016 Direct: 12.3 % Overall: 52.0 %
2019 Direct: 12.4 % Overall: 49.8 %
2022 Direct: 19.2 % Overall: 54.3 %


In [ ]:
Both measures of participation climbed around 2000, dropped substantially after 2005, and increased again by 2022.

SyntaxError: invalid syntax (1971732886.py, line 1)

***
## AI-use statement

Students must describe their own use of AI here, or state that they did not use it.

I used TECHi to learn how to plot the charts and to troubleshoot the kernel crashes. I tried its suggestions, but I did not determine what caused the crashes. Also used TECHi to understand different terms and give me some tips on how I could solve the tasks. 